In [ ]:
import numpy as np
from magtense.micromag import MicromagProblem
from typing import Optional
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import sys
sys.path.append("../../src/utils")
from utils import *
import time 

In [ ]:
#fname = "Grid_rasBase_5_nGrains_5_nRef_4_dG_3.75e-09.mat"
fname = "Grid_rasBase_5_nGrains_5_nRef_3_dG_5e-09.mat"

rng = np.random.default_rng(42)
problem = setup_grain_problem_from_matfile(fname, 
                                           cuda = True,
                                           cvode = False,
                                           rng=rng)

In [ ]:
#-------------- set external field values for hysteresis loop --------------
mu0 = 4 * np.pi * 1e-7
Hyst_dir = np.array([0, 0, 1])
h_ext_base = Hyst_dir / mu0 
steps = np.arange(1, -7.1, -0.1) 
#steps = np.arange(1, 0.0, -0.1) 
H_ext = np.zeros((len(steps), 4))
H_ext[:,0] = steps
H_ext[:, 1:4] = steps[:, np.newaxis] * h_ext_base  # External field values for hysteresis loop
#--------------------------------------------------------------------------

problem.exch_presize = 3 * problem.nt * len(steps)

In [ ]:
start_time = time.time()
res = problem.run_hysteresis(H_ext=H_ext)
end_time = time.time()
print(f"Hysteresis simulation took {end_time - start_time} seconds")

In [ ]:
M_out = res[1][0,:,:,:]

In [ ]:
mu0 = 4 * np.pi * 1e-7
alpha = 4000
gamma = 0.0
    #----------- inside grains -----------
Ms = 1.61/mu0 ; # 1.61 T
K0 = 4.3e6 ;    # J/m3
A0 = 7.7e-12 ;  # J/m3
    #------------------------------------

mesh_cart, GridInfo, mesh_params, iIn = load_matlab_struct(fname)

In [ ]:
volumes = GridInfo["Volumes"]
MxMean = np.zeros(len(steps))
MyMean = np.zeros(len(steps))
MzMean = np.zeros(len(steps))

vol_sum = np.sum(volumes)
Ms_vol = (problem.Ms* volumes).flatten()


for i in range(len(steps)):
    MxMean[i] = np.sum(Ms_vol * M_out[:,i,0] ) / vol_sum
    MyMean[i] = np.sum(Ms_vol * M_out[:,i,1] ) / vol_sum
    MzMean[i] = np.sum(Ms_vol * M_out[:,i,2] ) / vol_sum


In [ ]:
H_N = 2*K0/(Ms)


M = Hyst_dir[0] * MxMean + Hyst_dir[1] * MyMean + Hyst_dir[2] * MzMean
H = np.sign(H_ext[:,0]) * np.sqrt(H_ext[:,1]**2 + H_ext[:,2]**2 + H_ext[:,3]**2)*mu0 # What is this?

f = interp1d(M, H)     # H as function of M
Hc = float(f(0.0))

In [ ]:
import numpy as np

def max_energy_product(
    H: np.ndarray,
    M: np.ndarray,
    mu0: float = 4 * np.pi * 1e-7,
) -> tuple[float, float, float, float]:
    """
    Compute maximum energy product (BH)_max from hysteresis data
    using the discrete sample points (no interpolation).

    Parameters
    ----------
    H : array_like
        Applied field [A/m].
    M : array_like
        Magnetization [A/m].
    mu0 : float, optional
        Vacuum permeability [H/m].

    Returns
    -------
    BH_max : float
        Maximum energy product [J/m^3].
    H_at_max : float
        Field value at (BH)_max [A/m].
    B_at_max : float
        Flux density at (BH)_max [T].
    M_at_max : float
        Magnetization at (BH)_max [A/m].
    """
    H = np.asarray(H, dtype=float).ravel()
    M = np.asarray(M, dtype=float).ravel()

    # B(H) from SI relation
    B = mu0 * (H + M)

    # Second quadrant: H < 0, B > 0
    mask = (H < 0.0) & (B > 0.0)
    if not np.any(mask):
        raise ValueError("No points in the second quadrant (H<0, B>0) found.")

    Hq = H[mask]
    Bq = B[mask]
    Mq = M[mask]

    # Energy product (positive in 2nd quadrant)
    BH = -Bq * Hq  # J/m^3

    idx = int(np.argmax(BH))

    BH_max = BH[idx]
    H_at_max = Hq[idx]
    B_at_max = Bq[idx]
    M_at_max = Mq[idx]

    return BH_max, H_at_max, B_at_max, M_at_max




BH_max, H_star, B_star, M_star = max_energy_product(H, M)

print("BH_max = {:.2e} kJ/m^3".format(BH_max / 1e3))
print("H* = {:.3e} A/m".format(H_star))
print("B* = {:.3f} T".format(B_star))
print("M* = {:.3e} A/m".format(M_star))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(H, M / Ms, label='M/Ms')
ax.plot(Hc, 0, 'r*', markersize=10, label='Hc')
ax.plot(H_star, M_star / Ms, 'b*', markersize=10, label='$BH_{max}$')

ax.set_xlabel('H [ T ]')
ax.set_ylabel('M/Ms [ - ]')
ax.set_title('Hysteresis Loop')
ax.legend()
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()

#fig.savefig("hyst_01_cuda.png")